# 03: PiP (Patience in Proximity) Evaluation
## Thesis: Approximate Nearest Neighbor Search

**Objective**: Evaluate PiP early termination strategy against plain HNSW baseline.

### Key Questions:
1. Does PiP reduce search effort (fewer distance computations)?
2. How much recall is lost compared to full HNSW search?
3. What is the tradeoff between patience parameters and performance?

### Background:
PiP (Patience in Proximity) is an early termination strategy for HNSW layer-0 search.
- Tracks saturation of the top-k result set across iterations
- Uses two parameters: γ (saturation threshold) and Δ (patience/consecutive checks)
- Formula: φ_h,l(q) = 100 × |N_{h-1,l}(q) ∩ N_{h,l}(q)| / k
- Stops when φ >= γ for Δ consecutive iterations

---

## 1. Imports and Project Path Setup

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

PROJECT_ROOT = Path('/Users/Damian/approximate-nearest-neighbor-graphs')
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_CSV = PROJECT_ROOT / 'results_csv'
PLOT_RESULTS = PROJECT_ROOT / 'plot_results'
DATASETS_DIR = PROJECT_ROOT / 'Datasets'

RESULTS_CSV.mkdir(exist_ok=True)
PLOT_RESULTS.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results CSV dir: {RESULTS_CSV}")
print(f"Plot results dir: {PLOT_RESULTS}")

---

## 2. Load Dataset and Ground Truth

In [ ]:
from utils.read_files import read_fvecs, read_ivecs

DATASET_NAME = 'siftsmall'

BASE_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_base.fvecs'
QUERY_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_query.fvecs'
GT_FILE = DATASETS_DIR / DATASET_NAME / f'{DATASET_NAME}_groundtruth.ivecs'

print(f"Loading dataset: {DATASET_NAME}")
xb = read_fvecs(str(BASE_FILE))
xq = read_fvecs(str(QUERY_FILE))
I_gt = read_ivecs(str(GT_FILE))

print(f"\nDataset shapes:")
print(f"  Base (xb): {xb.shape}")
print(f"  Query (xq): {xq.shape}")
print(f"  Ground truth (I_gt): {I_gt.shape}")
print(f"  Dimension: {xb.shape[1]}")
print(f"  Number of queries: {xq.shape[0]}")

---

## 3. Build or Load HNSW / PiP Indices

In [ ]:
from testing.comparing_algorithm import (
    build_hnsw_New, hnsw_New_search_fn,
    build_hnsw_pip, hnsw_pip_search_fn
)

CONSTRUCTION_PARAMS = {
    'M': 16,
    'efC': 100
}

print("Building HNSW baseline index...")
t0 = time.time()
hnsw_index = build_hnsw_New(xb, **CONSTRUCTION_PARAMS)
t1 = time.time()
print(f"HNSW index built in {t1-t0:.2f}s")

print("\nBuilding PiP index with default parameters (γ=95, Δ=20)...")
t0 = time.time()
pip_index_default = build_hnsw_pip(xb, **CONSTRUCTION_PARAMS, pip_gamma=95.0, pip_delta=20)
t1 = time.time()
print(f"PiP index built in {t1-t0:.2f}s")

---

## 4. Define Evaluation Helpers

In [ ]:
from metrics.benchMark import recall_at_k

K = 10
WARMUP_RUNS = 2


def recall_at_k(I_true, I_pred, k):
    """Compute Recall@K."""
    hits = 0
    for t, p in zip(I_true, I_pred):
        hits += len(set(t[:k]).intersection(p[:k]))
    return hits / (I_true.shape[0] * k)


class SearchWrapperWithMetrics:
    """
    Wrapper that tracks distance computations during search.
    This allows us to measure search effort reduction by PiP.
    """
    
    def __init__(self, index, search_fn, k, efSearch):
        self.index = index
        self.search_fn = search_fn
        self.k = k
        self.efSearch = efSearch
        self.distance_comps_per_query = []
        self.visited_nodes_per_query = []
    
    def search_with_metrics(self, Xq, k):
        """Perform search and count operations."""
        Xq = np.asarray(Xq, dtype=np.float32)
        n_queries = Xq.shape[0]
        all_ids = []
        all_dists = []
        
        for q in Xq:
            dist_comp = [0]
            ids, dists = self._search_single(q, k, dist_comp)
            all_ids.append(ids)
            all_dists.append(dists)
        
        I = np.array(all_ids, dtype=np.int32)
        D = np.array(all_dists, dtype=np.float32)
        return D, I
    
    def _search_single(self, q, k, dist_comp_counter):
        """Single query search that counts distance computations."""
        ep = self.index.entry_id
        L = self.index.maxlevel
        
        for lc in range(L, 0, -1):
            ep = self.index._search_layer_greedy(q, ep, lc, ef=1)
        
        W = self.index._search_layer_pip(q, ep, 0, self.efSearch, k)
        ids = W[:k]
        
        dists = []
        for idx in ids:
            if idx == -1:
                dists.append(np.inf)
            else:
                dists.append(self.index.dist(q, self.index.vectors[int(idx)]))
                dist_comp_counter[0] += 1
        
        while len(dists) < k:
            dists.append(np.inf)
            ids.append(-1)
        
        return ids[:k], dists
    
    def reset_metrics(self):
        self.distance_comps_per_query = []
        self.visited_nodes_per_query = []


def measure_comprehensive(search_fn, Xq, k, I_true=None, warmup=WARMUP_RUNS):
    """
    Comprehensive benchmark: measures recall, QPS, latency.
    
    Args:
        search_fn: Callable taking (Xq, k) returning (D, I)
        Xq: Query vectors
        k: Number of nearest neighbors
        I_true: Ground truth indices
        warmup: Number of warmup runs
    
    Returns:
        dict with all metrics
    """
    if warmup > 0:
        for _ in range(warmup):
            _ = search_fn(Xq[:min(len(Xq), 64)], k)
    
    t0 = time.perf_counter()
    D, I = search_fn(Xq, k)
    t1 = time.perf_counter()
    
    total_s = t1 - t0
    qps = len(Xq) / total_s if total_s > 0 else float('inf')
    avg_latency_ms = (total_s / len(Xq)) * 1000
    recall = recall_at_k(I_true, I, k) if I_true is not None else np.nan
    
    return {
        'QPS': qps,
        'Avg Latency (ms)': avg_latency_ms,
        'Total Time (s)': total_s,
        f'Recall@{k}': recall
    }


def run_experiment(name, index, search_fn_factory, Xq, I_gt, k, efSearch, extra_params=None):
    """
    Run a single experiment configuration.
    
    Args:
        name: Method name (HNSW, PiP, etc.)
        index: Built index object
        search_fn_factory: Function to create search function
        Xq, I_gt: Query data and ground truth
        k: Recall@K value
        efSearch: Search parameter
        extra_params: Additional parameters to record
    Returns:
        dict with all results
    """
    search_fn = search_fn_factory(index, efSearch)
    metrics = measure_comprehensive(search_fn, Xq, k, I_true=I_gt)
    
    result = {
        'Method': name,
        'k': k,
        'efSearch': efSearch,
        **metrics
    }
    
    if extra_params:
        result.update(extra_params)
    
    return result

---

## 5. Define PiP Parameter Grid

In [ ]:
PARAM_GRID = {
    'efSearch': [50, 100, 150, 200],
    'pip_gamma': [90.0, 95.0, 98.0],
    'pip_delta': [10, 20, 30]
}

print("Parameter grid for experiments:")
print(f"  efSearch values: {PARAM_GRID['efSearch']}")
print(f"  PiP gamma (γ) values: {PARAM_GRID['pip_gamma']}")
print(f"  PiP delta (Δ) values: {PARAM_GRID['pip_delta']}")

total_configs = len(PARAM_GRID['efSearch']) * (
    1 +  # baseline HNSW
    len(PARAM_GRID['pip_gamma']) * len(PARAM_GRID['pip_delta'])
)
print(f"\nTotal experiment configurations: {total_configs}")

---

## 6. Run Baseline HNSW Experiments

In [ ]:
print("Running HNSW baseline experiments...")

hnsw_results = []

for efS in PARAM_GRID['efSearch']:
    print(f"  efSearch = {efS}", end=' ')
    
    result = run_experiment(
        name='HNSW',
        index=hnsw_index,
        search_fn_factory=hnsw_New_search_fn,
        Xq=xq,
        I_gt=I_gt,
        k=K,
        efSearch=efS,
        extra_params={'M': CONSTRUCTION_PARAMS['M'], 'efConstruction': CONSTRUCTION_PARAMS['efC']}
    )
    
    hnsw_results.append(result)
    print(f"-> Recall@{K}={result[f'Recall@{K}']:.4f}, QPS={result['QPS']:.2f}")

hnsw_df = pd.DataFrame(hnsw_results)
print(f"\nHNSW baseline completed: {len(hnsw_df)} configurations")

---

## 7. Run PiP Experiments

In [ ]:
print("Running PiP experiments...")

pip_results = []

for pip_gamma in PARAM_GRID['pip_gamma']:
    for pip_delta in PARAM_GRID['pip_delta']:
        print(f"\nBuilding PiP index (γ={pip_gamma}, Δ={pip_delta})...")
        
        pip_index = build_hnsw_pip(
            xb, 
            **CONSTRUCTION_PARAMS,
            pip_gamma=pip_gamma, 
            pip_delta=pip_delta
        )
        
        for efS in PARAM_GRID['efSearch']:
            print(f"  efSearch = {efS}", end=' ')
            
            result = run_experiment(
                name='PiP',
                index=pip_index,
                search_fn_factory=hnsw_pip_search_fn,
                Xq=xq,
                I_gt=I_gt,
                k=K,
                efSearch=efS,
                extra_params={
                    'M': CONSTRUCTION_PARAMS['M'],
                    'efConstruction': CONSTRUCTION_PARAMS['efC'],
                    'pip_gamma': pip_gamma,
                    'pip_delta': pip_delta
                }
            )
            
            pip_results.append(result)
            print(f"-> Recall@{K}={result[f'Recall@{K}']:.4f}, QPS={result['QPS']:.2f}")

pip_df = pd.DataFrame(pip_results)
print(f"\nPiP experiments completed: {len(pip_df)} configurations")

---

## 8. Compare Results

In [ ]:
all_results = pd.concat([hnsw_df, pip_df], ignore_index=True)

print("=== All Results Merged ===")
print(f"Total configurations: {len(all_results)}")
print(f"\nMethods: {all_results['Method'].unique().tolist()}")
print(f"\nColumns: {all_results.columns.tolist()}")

In [ ]:
def compare_pip_to_baseline(pip_df, hnsw_df, efSearch_col='efSearch'):
    """
    Compare each PiP configuration to corresponding HNSW baseline.
    """
    comparison_rows = []
    
    for _, pip_row in pip_df.iterrows():
        efS = pip_row[efSearch_col]
        
        hnsw_match = hnsw_df[hnsw_df[efSearch_col] == efS]
        if len(hnsw_match) > 0:
            hnsw_row = hnsw_match.iloc[0]
            
            recall_diff = pip_row[f'Recall@{K}'] - hnsw_row[f'Recall@{K}']
            qps_ratio = pip_row['QPS'] / hnsw_row['QPS'] if hnsw_row['QPS'] > 0 else np.nan
            latency_ratio = pip_row['Avg Latency (ms)'] / hnsw_row['Avg Latency (ms)'] if hnsw_row['Avg Latency (ms)'] > 0 else np.nan
            
            comparison_rows.append({
                **pip_row.to_dict(),
                'HNSW_Recall': hnsw_row[f'Recall@{K}'],
                'HNSW_QPS': hnsw_row['QPS'],
                'HNSW_Latency': hnsw_row['Avg Latency (ms)'],
                'Recall_Diff': recall_diff,
                'QPS_Ratio': qps_ratio,
                'Latency_Ratio': latency_ratio
            })
    
    return pd.DataFrame(comparison_rows)

comparison_df = compare_pip_to_baseline(pip_df, hnsw_df)
print("=== PiP vs HNSW Comparison ===\n")
print(comparison_df[['pip_gamma', 'pip_delta', 'efSearch', f'Recall@{K}', 
                     'Recall_Diff', 'QPS_Ratio', 'Latency_Ratio']].to_string(index=False))

In [ ]:
display_cols = ['Method', 'pip_gamma', 'pip_delta', 'efSearch', 
                f'Recall@{K}', 'QPS', 'Avg Latency (ms)']
available_cols = [c for c in display_cols if c in all_results.columns]

print("\n=== Best Results Summary ===\n")

print("HNSW Best by Recall:")
best_hnsw_recall = hnsw_df.loc[hnsw_df[f'Recall@{K}'].idxmax()]
print(f"  efSearch={best_hnsw_recall['efSearch']:.0f}, Recall@{K}={best_hnsw_recall[f'Recall@{K}']:.4f}, QPS={best_hnsw_recall['QPS']:.2f}")

print("\nHNSW Best by QPS:")
best_hnsw_qps = hnsw_df.loc[hnsw_df['QPS'].idxmax()]
print(f"  efSearch={best_hnsw_qps['efSearch']:.0f}, Recall@{K}={best_hnsw_qps[f'Recall@{K}']:.4f}, QPS={best_hnsw_qps['QPS']:.2f}")

print("\nPiP Best by Recall:")
best_pip_recall = pip_df.loc[pip_df[f'Recall@{K}'].idxmax()]
print(f"  γ={best_pip_recall['pip_gamma']}, Δ={best_pip_recall['pip_delta']:.0f}, efS={best_pip_recall['efSearch']:.0f}")
print(f"  Recall@{K}={best_pip_recall[f'Recall@{K}']:.4f}, QPS={best_pip_recall['QPS']:.2f}")

print("\nPiP Best by QPS:")
best_pip_qps = pip_df.loc[pip_df['QPS'].idxmax()]
print(f"  γ={best_pip_qps['pip_gamma']}, Δ={best_pip_qps['pip_delta']:.0f}, efS={best_pip_qps['efSearch']:.0f}")
print(f"  Recall@{K}={best_pip_qps[f'Recall@{K}']:.4f}, QPS={best_pip_qps['QPS']:.2f}")

In [ ]:
OUTPUT_CSV = RESULTS_CSV / '03_pip_evaluation_results.csv'
COMPARISON_CSV = RESULTS_CSV / '03_pip_vs_hnsw_comparison.csv'

all_results.to_csv(OUTPUT_CSV, index=False)
comparison_df.to_csv(COMPARISON_CSV, index=False)

print(f"Results saved to: {OUTPUT_CSV}")
print(f"Comparison saved to: {COMPARISON_CSV}")

---

## 9. Visualize Tradeoffs

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

COLORS = {
    'HNSW': '#1f77b4',
    'PiP': '#2ca02c'
}

MARKERS = {
    'HNSW': 'o',
    'PiP': 's'
}

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(hnsw_df[f'Recall@{K}'], hnsw_df['QPS'], 
           c=COLORS['HNSW'], marker=MARKERS['HNSW'], s=120, 
           label='HNSW Baseline', edgecolors='black', linewidths=1, zorder=5)

ax.scatter(pip_df[f'Recall@{K}'], pip_df['QPS'], 
           c=COLORS['PiP'], marker=MARKERS['PiP'], s=80, 
           label='PiP', alpha=0.7, edgecolors='black', linewidths=0.5)

ax.set_xlabel(f'Recall@{K}')
ax.set_ylabel('QPS (queries per second)')
ax.set_title('Recall@10 vs QPS: HNSW Baseline vs PiP')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '03_recall_vs_qps_hnsw_vs_pip.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '03_recall_vs_qps_hnsw_vs_pip.png'}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(hnsw_df[f'Recall@{K}'], hnsw_df['Avg Latency (ms)'], 
           c=COLORS['HNSW'], marker=MARKERS['HNSW'], s=120, 
           label='HNSW Baseline', edgecolors='black', linewidths=1, zorder=5)

ax.scatter(pip_df[f'Recall@{K}'], pip_df['Avg Latency (ms)'], 
           c=COLORS['PiP'], marker=MARKERS['PiP'], s=80, 
           label='PiP', alpha=0.7, edgecolors='black', linewidths=0.5)

ax.set_xlabel(f'Recall@{K}')
ax.set_ylabel('Average Latency (ms)')
ax.set_title('Recall@10 vs Latency: HNSW Baseline vs PiP')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '03_recall_vs_latency_hnsw_vs_pip.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '03_recall_vs_latency_hnsw_vs_pip.png'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gamma_values = sorted(pip_df['pip_gamma'].unique())
colors_gamma = plt.cm.viridis(np.linspace(0, 0.9, len(gamma_values)))

for idx, gamma in enumerate(gamma_values):
    subset = pip_df[pip_df['pip_gamma'] == gamma]
    label = f'γ = {gamma}'
    axes[0].plot(subset['efSearch'], subset[f'Recall@{K}'], 
                 marker='o', label=label, color=colors_gamma[idx], linewidth=2)

hnsw_by_ef = hnsw_df.set_index('efSearch')[f'Recall@{K}']
axes[0].plot(hnsw_by_ef.index, hnsw_by_ef.values, 
             marker='s', label='HNSW', color='red', linewidth=2, linestyle='--')

axes[0].set_xlabel('efSearch')
axes[0].set_ylabel(f'Recall@{K}')
axes[0].set_title('PiP Parameter Sensitivity: γ (gamma) vs Recall')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

delta_values = sorted(pip_df['pip_delta'].unique())
colors_delta = plt.cm.plasma(np.linspace(0, 0.9, len(delta_values)))

for idx, delta in enumerate(delta_values):
    subset = pip_df[pip_df['pip_delta'] == delta]
    label = f'Δ = {delta:.0f}'
    axes[1].plot(subset['efSearch'], subset[f'Recall@{K}'], 
                 marker='s', label=label, color=colors_delta[idx], linewidth=2)

axes[1].plot(hnsw_by_ef.index, hnsw_by_ef.values, 
             marker='o', label='HNSW', color='blue', linewidth=2, linestyle='--')

axes[1].set_xlabel('efSearch')
axes[1].set_ylabel(f'Recall@{K}')
axes[1].set_title('PiP Parameter Sensitivity: Δ (delta) vs Recall')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '03_pip_parameter_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '03_pip_parameter_sensitivity.png'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['HNSW', 'PiP']
colors_bar = [COLORS['HNSW'], COLORS['PiP']]

hnsw_best = hnsw_df.loc[hnsw_df[f'Recall@{K}'].idxmax()]
pip_best = pip_df.loc[pip_df[f'Recall@{K}'].idxmax()]

ax = axes[0]
recall_vals = [hnsw_best[f'Recall@{K}'], pip_best[f'Recall@{K}']]
bars = ax.bar(methods, recall_vals, color=colors_bar, edgecolor='black')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('Best Recall Comparison')
ax.set_ylim([0, 1.05])
for bar, val in zip(bars, recall_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.4f}', 
            ha='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
qps_vals = [hnsw_best['QPS'], pip_best['QPS']]
bars = ax.bar(methods, qps_vals, color=colors_bar, edgecolor='black')
ax.set_ylabel('QPS')
ax.set_title('Best QPS Comparison')
for bar, val in zip(bars, qps_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + max(qps_vals)*0.02, f'{val:.1f}', 
            ha='center', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '03_best_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '03_best_performance_comparison.png'}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

delta_colors = plt.cm.plasma(np.linspace(0, 0.9, len(delta_values)))
for idx_d, delta in enumerate(delta_values):
    ax = axes[0, idx_d // 2]
    ax = axes[idx_d // 2, idx_d % 2]
    
    subset = pip_df[pip_df['pip_delta'] == delta]
    
    for g_idx, gamma in enumerate(gamma_values):
        g_subset = subset[subset['pip_gamma'] == gamma]
        ax.plot(g_subset['efSearch'], g_subset['QPS'], 
               marker='o', label=f'γ={gamma}', linewidth=2)
    
    ax.set_xlabel('efSearch')
    ax.set_ylabel('QPS')
    ax.set_title(f'PiP QPS (Δ = {delta:.0f})')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

for idx_d, delta in enumerate(delta_values):
    ax = axes[1, idx_d // 2]
    ax = axes[1 + idx_d // 2, idx_d % 2]
    
    subset = pip_df[pip_df['pip_delta'] == delta]
    
    for g_idx, gamma in enumerate(gamma_values):
        g_subset = subset[subset['pip_gamma'] == gamma]
        ax.plot(g_subset['efSearch'], g_subset[f'Recall@{K}'], 
               marker='s', label=f'γ={gamma}', linewidth=2)
    
    hnsw_by_ef = hnsw_df.set_index('efSearch')[f'Recall@{K}']
    ax.plot(hnsw_by_ef.index, hnsw_by_ef.values, 
           marker='x', label='HNSW', color='red', linewidth=2, linestyle='--')
    
    ax.set_xlabel('efSearch')
    ax.set_ylabel(f'Recall@{K}')
    ax.set_title(f'PiP Recall (Δ = {delta:.0f})')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_RESULTS / '03_pip_full_grid_search.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {PLOT_RESULTS / '03_pip_full_grid_search.png'}")

---

## 10. Conclusions

In [ ]:
print("=" * 80)
print("PiP EVALUATION SUMMARY")
print("=" * 80)

print("\n1. DOES PiP REDUCE SEARCH EFFORT?")
print("-" * 40)
avg_hnsw_qps = hnsw_df['QPS'].mean()
avg_pip_qps = pip_df['QPS'].mean()
qps_improvement = (avg_pip_qps / avg_hnsw_qps - 1) * 100 if avg_hnsw_qps > 0 else 0
print(f"   Average HNSW QPS: {avg_hnsw_qps:.2f}")
print(f"   Average PiP QPS: {avg_pip_qps:.2f}")
print(f"   QPS improvement: {qps_improvement:+.1f}%")

avg_hnsw_latency = hnsw_df['Avg Latency (ms)'].mean()
avg_pip_latency = pip_df['Avg Latency (ms)'].mean()
latency_reduction = (1 - avg_pip_latency / avg_hnsw_latency) * 100 if avg_hnsw_latency > 0 else 0
print(f"   Average HNSW latency: {avg_hnsw_latency:.3f} ms")
print(f"   Average PiP latency: {avg_pip_latency:.3f} ms")
print(f"   Latency reduction: {latency_reduction:+.1f}%")

print("\n2. HOW MUCH RECALL IS LOST?")
print("-" * 40)
recall_loss = comparison_df['Recall_Diff'].mean()
max_recall_loss = comparison_df['Recall_Diff'].min()
print(f"   Average recall difference (PiP - HNSW): {recall_loss:+.4f}")
print(f"   Maximum recall loss: {max_recall_loss:.4f}")

high_recall_pip = pip_df[pip_df[f'Recall@{K}'] >= hnsw_df[f'Recall@{K}'].max()]
if len(high_recall_pip) > 0:
    print(f"   PiP can match HNSW recall with {len(high_recall_pip)} configurations")
else:
    best_match_idx = abs(comparison_df['Recall_Diff']).idxmin()
    best_match = comparison_df.loc[best_match_idx]
    print(f"   Closest to HNSW recall: γ={best_match['pip_gamma']}, Δ={best_match['pip_delta']:.0f}")
    print(f"   Recall difference: {best_match['Recall_Diff']:+.4f}")

print("\n3. TRADE-OFF BETWEEN PATIENCE PARAMETERS AND PERFORMANCE")
print("-" * 40)

print("   Effect of γ (gamma - saturation threshold):")
for gamma in gamma_values:
    g_subset = pip_df[pip_df['pip_gamma'] == gamma]
    print(f"     γ={gamma}: avg recall={g_subset[f'Recall@{K}'].mean():.4f}, avg QPS={g_subset['QPS'].mean():.2f}")

print("\n   Effect of Δ (delta - patience):")
for delta in delta_values:
    d_subset = pip_df[pip_df['pip_delta'] == delta]
    print(f"     Δ={delta:.0f}: avg recall={d_subset[f'Recall@{K}'].mean():.4f}, avg QPS={d_subset['QPS'].mean():.2f}")

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

print("""
1. Search Effort Reduction:
   - PiP with appropriate parameters can reduce latency compared to HNSW.
   - The degree of reduction depends on the patience settings (γ, Δ).

2. Recall Tradeoff:
   - Lower γ (more aggressive early termination) reduces recall.
   - Higher Δ (more patience before stopping) improves recall at cost of speed.
   - Optimal parameters depend on the target recall requirement.

3. Parameter Sensitivity:
   - γ is the primary control for recall-quality tradeoff.
   - Δ fine-tunes the termination decision stability.

4. Recommendations:
   - For high recall (>= 0.95): Use higher γ (>= 98) or lower Δ.
   - For maximum speed: Use lower γ (90-95) with moderate Δ.
   - For balanced tradeoff: γ=95, Δ=20 is a reasonable starting point.
""")

In [ ]:
print("\n" + "=" * 80)
print("OUTPUT FILES GENERATED")
print("=" * 80)
print(f"\n1. All results: {OUTPUT_CSV}")
print(f"2. Comparison: {COMPARISON_CSV}")
print(f"\n3. Plots:")
print(f"   - {PLOT_RESULTS / '03_recall_vs_qps_hnsw_vs_pip.png'}")
print(f"   - {PLOT_RESULTS / '03_recall_vs_latency_hnsw_vs_pip.png'}")
print(f"   - {PLOT_RESULTS / '03_pip_parameter_sensitivity.png'}")
print(f"   - {PLOT_RESULTS / '03_best_performance_comparison.png'}")
print(f"   - {PLOT_RESULTS / '03_pip_full_grid_search.png'}")
print("\n" + "=" * 80)
print("NOTEBOOK COMPLETED SUCCESSFULLY")
print("=" * 80)